# Lazy napari loading for mixed pyramid depths

This notebook keeps each input OME-TIFF separate so images with different numbers of pyramid levels can still be viewed together in napari.

Instead of forcing a shared multiscale stack across channels, it:
- opens each image lazily through `tiffslide`
- preserves that image's own pyramid depth
- adds each image to napari as its own multiscale layer


In [1]:
from pathlib import Path
from typing import Mapping

import napari
import xarray as xr


def _open_wsi_levels(path: str | Path) -> tuple[str, list[xr.DataArray], dict]:
    import tiffslide

    path = Path(path)
    slide = tiffslide.open_slide(str(path))
    zarr_store = slide.zarr_group.store
    zarr_img = xr.open_zarr(zarr_store, consolidated=False, mask_and_scale=False)
    levels = [zarr_img[str(k)] for k in sorted(zarr_img.keys(), key=int)]

    metadata = {
        "path": str(path),
        "dimensions": slide.dimensions,
        "level_count": slide.level_count,
        "level_dimensions": slide.level_dimensions,
        "level_downsamples": slide.level_downsamples,
        "properties": dict(slide.properties),
    }
    return path.stem, levels, metadata


def build_image_dict_from_folder(folder: str | Path) -> dict[str, str]:
    """
    Build a dictionary mapping `R<round>_<channel>` to file paths.

    Expected filename format:
        SLIDE-XXXX_<round>.0.X_R000_<dye>_<channel>_...ome.tif

    DAPI may have no `<channel>`, so that case is handled explicitly.
    """
    folder = Path(folder)
    image_dict: dict[str, str] = {}

    for file in sorted(folder.iterdir()):
        if not file.is_file():
            continue
        if not file.name.lower().endswith(".ome.tif"):
            continue

        parts = file.stem.split("_")
        try:
            round_str = parts[1]
            round_num = round_str.split(".")[0]

            dye = parts[3]
            channel = parts[4] if len(parts) > 4 and parts[4] else None

            channel_name = "DAPI" if dye.upper() == "DAPI" or not channel else channel
            key = f"R{round_num}_{channel_name}"
            image_dict[key] = str(file.resolve())
        except (IndexError, ValueError):
            print(f"Skipping file with unrecognized structure: {file.name}")

    return image_dict


def load_multiscale_layers_separately(
    image_files: Mapping[str, str],
) -> dict[str, dict]:
    """
    Open each image lazily and keep its pyramid separate.

    This avoids requiring all channels to have the same number of pyramid levels.
    """
    layers: dict[str, dict] = {}

    for channel_name, path in image_files.items():
        image_name, levels, metadata = _open_wsi_levels(path)
        layers[channel_name] = {
            "image_name": image_name,
            "path": path,
            "levels": [level.data for level in levels],
            "level_shapes": [tuple(level.shape[-2:]) for level in levels],
            "metadata": metadata,
        }

    return layers


def summarize_layers(layer_specs: Mapping[str, dict]) -> list[dict]:
    return [
        {
            "layer": layer_name,
            "level_count": len(spec["levels"]),
            "base_shape": spec["level_shapes"][0],
            "smallest_shape": spec["level_shapes"][-1],
            "path": spec["path"],
        }
        for layer_name, spec in layer_specs.items()
    ]


def add_layers_to_napari(
    layer_specs: Mapping[str, dict],
    *,
    blending: str = "additive",
) -> napari.Viewer:
    viewer = napari.Viewer()

    for layer_name, spec in layer_specs.items():
        viewer.add_image(
            spec["levels"],
            multiscale=True,
            name=layer_name,
            blending=blending,
            metadata={
                "source_path": spec["path"],
                "level_count": len(spec["levels"]),
                "level_shapes": spec["level_shapes"],
                "slide_metadata": spec["metadata"],
            },
        )

    return viewer


In [2]:
folder_path = Path(r"/mnt/e/SLIDE-0272_preview")
image_dict = build_image_dict_from_folder(folder_path)
image_dict


{'R1_DAPI': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_1.0.4_R000_DAPI__FINAL_F.ome.tif',
 'R10_Arg1-D4E3M-555-nimbus': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555-nimbus_FINAL_AFR_F.ome.tif',
 'R10_Arg1-D4E3M-555': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555_FINAL_AFR_F.ome.tif'}

In [3]:
layer_specs = load_multiscale_layers_separately(image_dict)
summarize_layers(layer_specs)


[{'layer': 'R1_DAPI',
  'level_count': 7,
  'base_shape': (60899, 59212),
  'smallest_shape': (951, 925),
  'path': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_1.0.4_R000_DAPI__FINAL_F.ome.tif'},
 {'layer': 'R10_Arg1-D4E3M-555-nimbus',
  'level_count': 1,
  'base_shape': (60899, 59212),
  'smallest_shape': (60899, 59212),
  'path': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555-nimbus_FINAL_AFR_F.ome.tif'},
 {'layer': 'R10_Arg1-D4E3M-555',
  'level_count': 7,
  'base_shape': (60899, 59212),
  'smallest_shape': (951, 925),
  'path': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555_FINAL_AFR_F.ome.tif'}]

In [4]:
viewer = add_layers_to_napari(layer_specs)
viewer


Viewer(mouse_move_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x7e177fa5fec0>], mouse_drag_callbacks=[<function drag_to_zoom at 0x7e177fa78040>], mouse_double_click_callbacks=[<function double_click_to_zoom at 0x7e177fa5ff60>], camera=Camera(center=(0.0, 30449.0, 29605.5), zoom=0.014694822575083335, angles=(0.0, 0.0, 0.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(1.0, 1.0), viewbox=None, scaled=True, size=1.0, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=2, ndisplay=2, order=(0, 1), axis_labels=('-2', '-1'), rollable=(True, True), range=(RangeTuple(start=0.0, stop=60898.0, step=1.0), RangeTuple(start=0.0, stop=59211.0, step=1.0)), margin_left=(0.0, 0.0), margin_right=(0.0, 0.0), point=(30449.0, 29605.0), units=(<Unit('pixel')>, <Unit('pixel')>), last_used=0), grid=GridCanvas(stride=1